# 03 - Final Experiment and Error Analysis
Complete end-to-end pipeline evaluation including error analysis and confidence calibration.

In [ ]:
import sys
sys.path.insert(0, '..')

from src.data.ingestion import DataIngestor
from src.data.preprocessing import DataPreprocessor
from src.features.engineering import FeatureEngineer
from src.risk.baseline import BaselineRiskModel
from src.risk.risk_engine import MultiFactorRiskModel
from src.risk.confidence import ConfidenceEvaluator
from src.optimisation.planner import OutreachPlanner
from src.fairness.bias_detection import FairnessAuditor
from src.communication.message_generator import MessageGenerator
from src.evaluation.error_analysis import find_high_confidence_errors, summarise_errors_by_district
import matplotlib.pyplot as plt
import pandas as pd

# Full pipeline (with higher missing rate to stress-test)
valid_df, errors = DataIngestor.load_synthetic_data(num_records=300, missing_rate=0.15)
raw_df = valid_df.copy()

preprocessor = DataPreprocessor()
preprocessor.fit(valid_df)
processed_df = preprocessor.transform(valid_df)
featured_df = FeatureEngineer().generate_features(processed_df)
risk_df = MultiFactorRiskModel().calculate_risk(BaselineRiskModel().calculate_risk(featured_df))
confident_df = ConfidenceEvaluator().evaluate(raw_df, risk_df)
planned_df = OutreachPlanner().plan_outreach(confident_df)
final_df = MessageGenerator().generate_messages(planned_df)

print(f'Pipeline complete. Records: {len(final_df)}, Rejected at validation: {len(errors)}')
print(f'Selected for outreach: {int(final_df.selected_for_outreach.sum())}')
print(f'Awaiting manual review (low confidence): {int((final_df.fallback_status=="MANUAL_REVIEW").sum())}')

## High-Confidence False Negatives

These are the **most dangerous errors**: areas the model rated LOW risk with HIGH confidence, but which are actually high-risk according to our oracle benchmark.

In [ ]:
errors_df = find_high_confidence_errors(final_df)
print(f'Dangerous False Negatives: {len(errors_df)}')

if not errors_df.empty:
    print('\nDistrict breakdown:')
    print(summarise_errors_by_district(errors_df).to_string(index=False))
    
    # Plot if errors found
    district_errors = summarise_errors_by_district(errors_df)
    district_errors.plot(kind='bar', x='district', y='error_count', color='#FF4B4B', legend=False)
    plt.title('High-Confidence False Negatives by District')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig('../data/processed/error_analysis.png', dpi=150)
    plt.show()
else:
    print('No high-confidence false negatives found.')

## Confidence Score Calibration

In [ ]:
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Histogram of confidence scores
final_df['confidence_score'].hist(bins=20, ax=axes[0], color='#00CC96', edgecolor='black')
axes[0].set_title('Confidence Score Distribution')
axes[0].set_xlabel('Confidence Score')
axes[0].axvline(0.6, color='orange', linestyle='--', label='Warning threshold')
axes[0].axvline(0.85, color='green', linestyle='--', label='Auto-approve threshold')
axes[0].legend()

# Confidence vs Risk score scatter
sc = axes[1].scatter(final_df['confidence_score'], final_df['multi_factor_risk_score'],
                     c=final_df['selected_for_outreach'].astype(int), cmap='RdYlGn', alpha=0.6)
axes[1].set_title('Confidence vs Risk (Green=Selected, Red=Not Selected)')
axes[1].set_xlabel('Confidence Score')
axes[1].set_ylabel('Multi-Factor Risk Score')

plt.tight_layout()
plt.savefig('../data/processed/confidence_calibration.png', dpi=150)
plt.show()

## Fairness Analysis

In [ ]:
warnings = FairnessAuditor().audit_plan(final_df)
if warnings:
    for w in warnings:
        print(f'⚠️  {w["message"]}')
else:
    print('✅  No fairness violations detected.')

## Final Conclusions

| Aspect | Baseline | Multi-Factor |
|--------|----------|--------------|
| Features used | Temperature only | Temperature + Built Env + Vulnerability + Services |
| Risk differentiation | Broad categories | Granular, weighted |
| Fairness audit | N/A | ✅ Coverage parity enforced |
| Confidence assessment | N/A | ✅ Data completeness tracked |
| Human override | N/A | ✅ Audit-logged overrides |

**Conclusion:** The Multi-Factor model is strictly superior for public health outreach planning because it accounts for social and environmental context, not just meteorological extremes.

In [ ]:
print('Final experiment complete. All outputs saved to data/processed/')